# FraudGuard — Pipeline ML Final (tout-en-un)

**Objectif de ce notebook** : produire un livrable simple pour l'étudiant backend — **2 fichiers seulement** :
1. `fraudguard_pipeline.joblib` — un seul fichier qui contient TOUT (modèle, scaler, colonnes, seuil, tables de fréquence, catégories/régions connues)
2. `predire_fraude.py` — un script avec une seule fonction à appeler : `predire_fraude(transaction)`

Ce notebook reprend le travail déjà fait dans les notebooks 1 et 2 (EDA, comparaison de modèles), mais condense tout le pipeline final en un seul endroit, propre et exécutable d'un bout à l'autre.

⚠️ Place ce notebook dans le même dossier que `fraudTest_senegal.csv`, puis fais **Run All Cells**.

## 1. Imports

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

print("Librairies chargées.")

Librairies chargées.


## 2. Chargement des données brutes

In [3]:
df = pd.read_csv(r"C:\Users\LUXE TECH\Desktop\Detection-fraud\fraudTest_senegal.csv")
print("Dimensions :", df.shape)
df.head(2)

Dimensions : (555719, 24)


,id,date_transaction,num_carte,commercant,categorie,montant_fcfa,prenom,nom,genre,rue,...,longitude_client,population_ville,profession,date_naissance,num_transaction,temps_unix,latitude_commercant,longitude_commercant,est_fraude,pays
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,soins_personnels,1601.6,Aminata,Kane,M,351 Darlene Green,...,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0,Sénégal
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,soins_personnels,16710.4,Aïssatou,Ly,F,3638 Marsh Union,...,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0,Sénégal


## 3. Feature engineering

On calcule l'âge, l'heure/jour de transaction, et la distance client-commerçant — exactement comme dans le notebook 1.

In [4]:
df['date_transaction'] = pd.to_datetime(df['date_transaction'])
df['date_naissance'] = pd.to_datetime(df['date_naissance'])

df['age_client'] = (df['date_transaction'] - df['date_naissance']).dt.days // 365
df['heure_transaction'] = df['date_transaction'].dt.hour
df['jour_semaine'] = df['date_transaction'].dt.dayofweek
df['est_weekend'] = df['jour_semaine'].isin([5,6]).astype(int)

def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * np.arcsin(np.sqrt(a)) * 6371

df['distance_client_commercant_km'] = haversine_distance(
    df['latitude_client'], df['longitude_client'],
    df['latitude_commercant'], df['longitude_commercant']
)

print("Features créées.")

Features créées.


## 4. Tables de référence (à garder pour le backend)

On calcule les tables de fréquence et les listes de catégories/régions **avant** toute transformation — ce sont elles qui permettront de traiter une nouvelle transaction plus tard, exactement comme ici.

In [5]:
freq_commercant = df['commercant'].value_counts(normalize=True).to_dict()
freq_ville = df['ville'].value_counts(normalize=True).to_dict()
freq_profession = df['profession'].value_counts(normalize=True).to_dict()
categories_connues = sorted(df['categorie'].unique().tolist())
regions_connues = sorted(df['region'].unique().tolist())

print("Catégories :", categories_connues)
print("Régions :", regions_connues)

Catégories : ['achats', 'achats_en_ligne', 'auto_transport', 'courses', 'courses_en_ligne', 'divers', 'divers_en_ligne', 'divertissement', 'enfants_animaux', 'maison', 'restauration', 'sante_sport', 'soins_personnels', 'voyage']
Régions : ['Dakar', 'Diourbel', 'Fatick', 'Kaolack', 'Kolda', 'Louga', 'Matam', 'Saint-Louis', 'Sédhiou', 'Tambacounda', 'Thiès', 'Ziguinchor']


## 5. Nettoyage et encodage

Suppression des colonnes PII/inutiles, encodage du genre, One-Hot pour catégorie/région, encodage par fréquence pour commerçant/ville/profession.

In [6]:
colonnes_a_supprimer = [
    'id', 'num_carte', 'num_transaction', 'prenom', 'nom', 'rue',
    'date_transaction', 'date_naissance', 'temps_unix', 'pays',
    'code_postal', 'latitude_client', 'longitude_client',
    'latitude_commercant', 'longitude_commercant'
]
df_model = df.drop(columns=colonnes_a_supprimer)

df_model['genre'] = df_model['genre'].map({'M': 0, 'F': 1})
df_model = pd.get_dummies(df_model, columns=['categorie', 'region'], drop_first=True)

for col, freq in [('commercant', freq_commercant), ('ville', freq_ville), ('profession', freq_profession)]:
    df_model[col + '_freq'] = df_model[col].map(freq)
    df_model.drop(columns=[col], inplace=True)

print("Dimensions après encodage :", df_model.shape)

Dimensions après encodage : (555719, 36)


## 6. Split train/test et standardisation

In [7]:
X = df_model.drop(columns=['est_fraude'])
y = df_model['est_fraude']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

colonnes_numeriques = ['montant_fcfa', 'age_client', 'distance_client_commercant_km',
                        'population_ville', 'commercant_freq', 'ville_freq', 'profession_freq']

scaler = StandardScaler()
X_train[colonnes_numeriques] = scaler.fit_transform(X_train[colonnes_numeriques])
X_test[colonnes_numeriques] = scaler.transform(X_test[colonnes_numeriques])

colonnes_modele = list(X_train.columns)
print("Train :", X_train.shape, "| Test :", X_test.shape)

Train : (444575, 35) | Test : (111144, 35)


## 7. SMOTE + entraînement du modèle final

On utilise directement les meilleurs hyperparamètres déjà trouvés lors de l'optimisation du notebook 2 (RandomizedSearchCV), pour ne pas relancer une recherche longue.

In [8]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

best_model = XGBClassifier(
    n_estimators=300, max_depth=10, learning_rate=0.2,
    subsample=1.0, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
best_model.fit(X_train_sm, y_train_sm)

print("Modèle entraîné.")

Modèle entraîné.


## 8. Évaluation

In [9]:
y_proba = best_model.predict_proba(X_test)[:,1]
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['Légitime', 'Fraude'], digits=3))
print("ROC-AUC :", round(roc_auc_score(y_test, y_proba), 4))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

    Légitime      0.999     1.000     0.999    110715
      Fraude      0.900     0.800     0.847       429

    accuracy                          0.999    111144
   macro avg      0.950     0.900     0.923    111144
weighted avg      0.999     0.999     0.999    111144

ROC-AUC : 0.9933
[[110677     38]
 [    86    343]]


## 9. Choix du seuil métier (Recall cible ≈ 85%)

In [10]:
precisions, recalls, seuils = precision_recall_curve(y_test, y_proba)
idx = np.argmin(np.abs(recalls[:-1] - 0.85))
seuil_decision = float(seuils[idx])

print(f"Seuil choisi : {seuil_decision:.4f}")
print(f"Recall : {recalls[idx]:.3f} | Precision : {precisions[idx]:.3f}")

Seuil choisi : 0.1126
Recall : 0.851 | Precision : 0.782


## 10. Export : UN SEUL fichier pour tout

Au lieu de 9 fichiers séparés, on regroupe tout dans un seul dictionnaire Python, sauvegardé en un seul `.joblib`. Beaucoup plus simple à transmettre et à charger côté backend.

In [11]:
pipeline_complet = {
    "model": best_model,
    "scaler": scaler,
    "colonnes_modele": colonnes_modele,
    "colonnes_numeriques": colonnes_numeriques,
    "seuil_decision": seuil_decision,
    "freq_commercant": freq_commercant,
    "freq_ville": freq_ville,
    "freq_profession": freq_profession,
    "categories_connues": categories_connues,
    "regions_connues": regions_connues,
}

joblib.dump(pipeline_complet, "fraudguard_pipeline.joblib")
print("fraudguard_pipeline.joblib exporté avec succès — un seul fichier contient tout.")

fraudguard_pipeline.joblib exporté avec succès — un seul fichier contient tout.


## 11. Génération automatique du script `predire_fraude.py`

Cette cellule écrit directement le fichier Python final sur le disque — tu n'as rien à copier-coller.

In [12]:
code_script = '''"""
predire_fraude.py
==================
Script backend pour scorer une transaction avec le modèle FraudGuard.

UTILISATION :
    from predire_fraude import predire_fraude

    transaction = {
        "montant_fcfa": 45000,
        "genre": "F",
        "ville": "Dakar",
        "region": "Dakar",
        "population_ville": 1200000,
        "profession": "Sales professional, IT",
        "categorie": "achats_en_ligne",
        "commercant": "fraud_Kirlin and Sons",
        "date_transaction": "2026-08-11 14:30:00",
        "date_naissance": "1990-03-19",
        "latitude_client": 14.7167,
        "longitude_client": -17.4677,
        "latitude_commercant": 14.7500,
        "longitude_commercant": -17.5000,
    }

    resultat = predire_fraude(transaction)
    # {"est_fraude": True/False, "probabilite": 0.83, "seuil_utilise": 0.113}

FICHIER REQUIS DANS LE MEME DOSSIER : fraudguard_pipeline.joblib (un seul fichier, tout est dedans)
"""

import os
import numpy as np
import pandas as pd
import joblib

_PIPELINE = None


def _charger():
    global _PIPELINE
    chemin = os.path.join(os.environ.get("ML_MODELS_DIR", "."), "fraudguard_pipeline.joblib")
    _PIPELINE = joblib.load(chemin)


def _haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * np.arcsin(np.sqrt(a)) * 6371


def predire_fraude(transaction: dict) -> dict:
    if _PIPELINE is None:
        _charger()

    p = _PIPELINE
    t = dict(transaction)

    date_transaction = pd.to_datetime(t["date_transaction"])
    date_naissance = pd.to_datetime(t["date_naissance"])

    age_client = (date_transaction - date_naissance).days // 365
    heure_transaction = date_transaction.hour
    jour_semaine = date_transaction.dayofweek
    est_weekend = int(jour_semaine in [5, 6])
    distance_km = _haversine(t["latitude_client"], t["longitude_client"],
                              t["latitude_commercant"], t["longitude_commercant"])
    genre_encode = 1 if t["genre"] == "F" else 0

    commercant_freq = p["freq_commercant"].get(t["commercant"], 0.0)
    ville_freq = p["freq_ville"].get(t["ville"], 0.0)
    profession_freq = p["freq_profession"].get(t["profession"], 0.0)

    ligne = {col: 0 for col in p["colonnes_modele"]}
    valeurs = {
        "montant_fcfa": t["montant_fcfa"], "genre": genre_encode,
        "population_ville": t["population_ville"], "age_client": age_client,
        "heure_transaction": heure_transaction, "jour_semaine": jour_semaine,
        "est_weekend": est_weekend, "distance_client_commercant_km": distance_km,
        "commercant_freq": commercant_freq, "ville_freq": ville_freq,
        "profession_freq": profession_freq,
    }
    for col, val in valeurs.items():
        if col in ligne:
            ligne[col] = val

    col_cat = f"categorie_{t[\'categorie\']}"
    if col_cat in ligne:
        ligne[col_cat] = 1
    col_reg = f"region_{t[\'region\']}"
    if col_reg in ligne:
        ligne[col_reg] = 1

    X = pd.DataFrame([ligne])[p["colonnes_modele"]]
    X[p["colonnes_numeriques"]] = p["scaler"].transform(X[p["colonnes_numeriques"]])

    proba = float(p["model"].predict_proba(X)[:, 1][0])
    est_fraude = bool(proba >= p["seuil_decision"])

    return {
        "est_fraude": est_fraude,
        "probabilite": round(proba, 4),
        "seuil_utilise": float(p["seuil_decision"]),
    }
'''

with open("predire_fraude.py", "w", encoding="utf-8") as f:
    f.write(code_script)

print("predire_fraude.py généré avec succès.")

predire_fraude.py généré avec succès.


## 12. Test de bout en bout

On vérifie que tout fonctionne, en simulant une vraie transaction reçue par l'API (donc sous forme de dictionnaire brut, pas de DataFrame).

In [13]:
# On force le rechargement pour être sûr de tester le fichier fraîchement généré
import importlib
import predire_fraude as pf
importlib.reload(pf)

transaction_exemple = {
    "montant_fcfa": 45000,
    "genre": "F",
    "ville": df['ville'].iloc[0],
    "region": df['region'].iloc[0],
    "population_ville": int(df['population_ville'].iloc[0]),
    "profession": df['profession'].iloc[0],
    "categorie": df['categorie'].iloc[0],
    "commercant": df['commercant'].iloc[0],
    "date_transaction": "2026-08-11 14:30:00",
    "date_naissance": "1990-03-19",
    "latitude_client": float(df['latitude_client'].iloc[0]),
    "longitude_client": float(df['longitude_client'].iloc[0]),
    "latitude_commercant": float(df['latitude_commercant'].iloc[0]),
    "longitude_commercant": float(df['longitude_commercant'].iloc[0]),
}

resultat = pf.predire_fraude(transaction_exemple)
print("Résultat :", resultat)

Résultat : {'est_fraude': False, 'probabilite': 0.0, 'seuil_utilise': 0.11258848011493683}


---
## ✅ Livrable final pour l'étudiant backend

**Seulement 2 fichiers à transmettre :**
1. `fraudguard_pipeline.joblib` — contient tout (modèle, scaler, colonnes, seuil, tables de fréquence)
2. `predire_fraude.py` — une seule fonction à importer et appeler

**Ce qu'il doit faire, en 3 lignes :**
```python
from predire_fraude import predire_fraude
resultat = predire_fraude(transaction_brute_recue_par_l_api)
# {"est_fraude": True/False, "probabilite": 0.83, "seuil_utilise": 0.113}
```

Placer les 2 fichiers dans `ml_models/` (dossier déjà prévu dans son `settings.py` via `ML_MODELS_DIR`). Rien d'autre à configurer, rien à recoder.